# Exp8.0.5 — Frozen backbone phase readout

Aggregation-only notebook. It reads finalized CSV/JSON artifacts and does **not** train or refit any model.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
while repo != repo.parent and not (repo / 'pyproject.toml').exists():
    repo = repo.parent
base = repo / 'notebooks' / 'artifacts' / 'experiment_8_0_5_frozen_backbone_phase_readout' / 'frozen_backbone_phase_readout_v1'
manifest = json.loads((base / 'manifest.json').read_text())
method_summary = pd.read_csv(base / 'method_summary.csv')
paired_summary = pd.read_csv(base / 'paired_delta_summary.csv')
branch_summary = pd.read_csv(base / 'branch_ablation_summary.csv')
phase_summary = pd.read_csv(base / 'phase_shift_summary.csv')
history = pd.read_csv(base / 'history_runs.csv')
manifest


## Native frozen-readout comparison


In [ ]:
cols = ['method', 'test_ba_mean', 'test_ba_std', 'val_ba_mean', 'best_epoch_mean', 'parameter_count_mean']
display(method_summary[cols].sort_values('test_ba_mean', ascending=False))

plot_df = method_summary.set_index('method')
ax = plot_df['test_ba_mean'].plot(kind='bar', yerr=plot_df['test_ba_std'], capsize=4)
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Exp8.0.5 frozen-backbone readouts')
plt.tight_layout()
plt.show()


## Paired seed deltas


In [ ]:
display(paired_summary)
primary = paired_summary[paired_summary['comparison'] == 'true_phase_vs_destroyed_phase']
primary


## L1/L2 branch contribution diagnostics


In [ ]:
test_branch = branch_summary[branch_summary['split'] == 'test'].copy()
cols = ['method', 'full_ba_mean', 'l1_only_ba_mean', 'l2_only_ba_mean', 'l1_removal_drop_mean', 'l2_removal_drop_mean', 'full_equals_branch_sum_max_error_mean']
display(test_branch[cols])

branch_plot = test_branch.set_index('method')[['full_ba_mean', 'l1_only_ba_mean', 'l2_only_ba_mean']]
ax = branch_plot.plot(kind='bar')
ax.set_ylabel('Balanced accuracy')
ax.set_title('Full vs L1-only vs L2-only, no retraining')
plt.tight_layout()
plt.show()


## True-phase circular-shift diagnostic

`shift_bins=0` is the correctly aligned phase assignment. All other points keep the trained head fixed and deliberately misalign L1 phase blocks.


In [ ]:
display(phase_summary[['shift_bins', 'shift_ms', 'test_ba_mean', 'test_ba_std']])
curve = phase_summary.sort_values('shift_bins')
plt.errorbar(curve['shift_bins'], curve['test_ba_mean'], yerr=curve['test_ba_std'], marker='o')
plt.xlabel('Circular phase shift (250-ms bins)')
plt.ylabel('Test balanced accuracy')
plt.title('Phase alignment sensitivity')
plt.tight_layout()
plt.show()


## Method-level training curves


In [ ]:
mean_history = history.groupby(['method', 'epoch'], as_index=False)[['train_ba', 'val_ba', 'train_loss', 'val_loss']].mean()
for method, frame in mean_history.groupby('method'):
    plt.plot(frame['epoch'], frame['val_ba'], label=method)
plt.xlabel('Epoch')
plt.ylabel('Mean validation BA')
plt.title('Readout-only training, averaged across seeds')
plt.legend()
plt.tight_layout()
plt.show()
